# Notes
Helpful info from udemy course: 
https://github.com/mrdbourke/pytorch-deep-learning/blob/main/01_pytorch_workflow.ipynb

# GitHub Repository:
https://github.com/mpennino/Future_DW_NO3

In [168]:
# Import libraries
import pandas as pd
import torch
import torch.nn as nn
import numpy as np
import random


import pyarrow as pa
import pyarrow.parquet as pq


In [169]:
# Make device agnostic code
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cpu'

In [170]:
# Load Observation Dataset
# DATA = readRDS(paste0(strap_dir,'Data/Models/RF_bi_model_All_DATA_all_vars_','Trends_Conc_PWS_GW_05to20', '.rds'))
#future_dir = 'C:/Users/MPennino/OneDrive - Environmental Protection Agency (EPA)/Projects/StRAPs/StRAP4/SSWR.405.1_Future_DW/Data/'
future_dir = 'C:/Users/MPennino/OneDrive - Environmental Protection Agency (EPA)/Projects/OASES/Data/Future_NO3/'


#dataset1 = 'Dataset_RF_Model_SW_COMIDS.csv'
dataset1 = 'Dataset_RF_Model_GW_COMIDS.csv'

input_data_ = pd.read_csv(future_dir+dataset1)
input_data_.head(3)

,COMID,viol_freq,PopDen2010Cat,PctCrop2019Cat,AgKffactCat,precip9120cat,tmean9120cat,permcat,HydrlCondCat,RockNCat,N_TW2012Cat,N_Surp_kgsqkm_2017cat,WtDepCat,ElevCat,Fe2O3Cat,SandCat,Hillslope_PctCat,Viol_Class
0,-504163,0,12.6213,36.43,0.0489,1390.868070,20.671089,17.240000,0.0557,83.7932,5.863261,5060.694776,121.2300,17.0535,0.7600,73.2634,0.859481,0
1,-504162,0,37.8686,11.85,0.0392,1383.782884,20.638637,17.243133,0.0671,89.8335,6.016214,5283.705054,121.2034,17.2614,0.7617,73.3484,0.938520,0
2,-504156,0,43.3216,27.39,0.0605,1394.731896,20.595718,27.154088,0.0557,84.6277,6.224132,6281.618355,159.5239,13.5562,0.7600,85.8590,0.895041,0


In [171]:
input_data_.shape


(115839, 18)

In [172]:
input_data1 = input_data_.drop(columns=['viol_freq']) 

names_list = input_data1.columns.tolist()
print(names_list) 

['COMID', 'PopDen2010Cat', 'PctCrop2019Cat', 'AgKffactCat', 'precip9120cat', 'tmean9120cat', 'permcat', 'HydrlCondCat', 'RockNCat', 'N_TW2012Cat', 'N_Surp_kgsqkm_2017cat', 'WtDepCat', 'ElevCat', 'Fe2O3Cat', 'SandCat', 'Hillslope_PctCat', 'Viol_Class']


In [173]:
# Remove extra fields
# For Surface Water Dataset (,'AgDrain_pctWs','Hillslope_PctWs','BFIWs')
#input_data = input_data_.drop(columns=['HUC12','viol_freq','PopDen2010Ws','WaterInputWs','wdrw_LDWs','FertWs','CBNFWs','ManureWs','Septic_km2Cat']) 
input_data = input_data_.drop(columns=['COMID','viol_freq']) 

# For Groundwater Dataset
#input_data = input_data_.drop(columns=['HUC12','viol_freq','PopDen2010Cat','AgKffactCat','Septic_km2Cat','AgDrain_pctCat','WaterInputCat','wdrw_LDCat','BFICat','Hillslope_PctCat']) 

input_data.head(3)


,PopDen2010Cat,PctCrop2019Cat,AgKffactCat,precip9120cat,tmean9120cat,permcat,HydrlCondCat,RockNCat,N_TW2012Cat,N_Surp_kgsqkm_2017cat,WtDepCat,ElevCat,Fe2O3Cat,SandCat,Hillslope_PctCat,Viol_Class
0,12.6213,36.43,0.0489,1390.868070,20.671089,17.240000,0.0557,83.7932,5.863261,5060.694776,121.2300,17.0535,0.7600,73.2634,0.859481,0
1,37.8686,11.85,0.0392,1383.782884,20.638637,17.243133,0.0671,89.8335,6.016214,5283.705054,121.2034,17.2614,0.7617,73.3484,0.938520,0
2,43.3216,27.39,0.0605,1394.731896,20.595718,27.154088,0.0557,84.6277,6.224132,6281.618355,159.5239,13.5562,0.7600,85.8590,0.895041,0


In [174]:
input_data.shape,input_data_.shape

((115839, 16), (115839, 18))

In [175]:
# Calculate accuracy (a classification metric)
def accuracy_fn(y_true, y_pred):
    correct = torch.eq(y_true, y_pred).sum().item() # torch.eq() calculates where two tensors are equal
    acc = (correct / len(y_pred)) * 100 
    return acc

# Create Balanced Dataset


In [176]:
print(input_data['Viol_Class'].value_counts())

Viol_Class
0    114566
1      1273
Name: count, dtype: int64


In [177]:
# Save Preditor Data for SHAP Analysis
X_input_data = input_data.drop(columns=['Viol_Class']).values
X_input_data.shape


(115839, 15)

In [178]:
# Convert to tensor data
X_input_data = torch.from_numpy(X_input_data).type(torch.float)
X_input_data.shape,X_input_data.dtype

(torch.Size([115839, 15]), torch.float32)

In [179]:
min_size = input_data['Viol_Class'].value_counts().min()
min_size

np.int64(1273)

In [180]:
# Find the size of the smallest class
min_size = input_data['Viol_Class'].value_counts().min()

min_size  = min_size * 10

# Sample exactly 'min_size' elements from each binary group
#balanced_df = input_data.groupby('Viol_Class').sample(n=min_size, random_state=42).reset_index(drop=True)

# If increasing the min_size, you can use the 'replace=True' argument to allow for sampling with replacement
balanced_df = input_data.groupby('Viol_Class').sample(n=min_size, random_state=42, replace=True).reset_index(drop=True)

print(balanced_df['Viol_Class'].value_counts())

Viol_Class
0    12730
1    12730
Name: count, dtype: int64


# Transform data to torch tensor


In [181]:
# Convert to tensors and split into train and test sets
from sklearn.model_selection import train_test_split
#X = input_data.drop(columns=['Viol_Class']).values # when use this the model just predicts the majority class
#y = input_data['Viol_Class'].values
X = balanced_df.drop(columns=['Viol_Class']).values
y = balanced_df['Viol_Class'].values

# Turn data into tensors
X = torch.from_numpy(X).type(torch.float)
y = torch.from_numpy(y).type(torch.float)

# Make a copy to use later
X_full = X.clone()
y_full = y.clone()

# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, 
                                                    y, 
                                                    test_size=0.2,
                                                    random_state=2
)

#X_train[:5], y_train[:5]

In [182]:
type(X_train)
# print(y_train.unique(return_counts=True)),
# print(y_test.unique(return_counts=True)),

torch.Tensor

In [183]:
#X.dtype, y.dtype, X.size(), y.size()

# Create NN Model

In [184]:
nrows = X_train.size()[0]
ncols = X_train.size()[1]
nrows,ncols

(20368, 15)

In [185]:
# Build SW model with non-linear activation function
# from torch import nn

# featureNum = 5
# class BinaryClassifier(nn.Module):
#     def __init__(self):
#         super().__init__()
#         self.layer_1 = nn.Linear(in_features=ncols, out_features=featureNum) 
#         self.layer_2 = nn.Linear(in_features=featureNum, out_features=featureNum)
#         self.layer_3 = nn.Linear(in_features=featureNum, out_features=1)
#         self.relu = nn.ReLU() # <- add in ReLU activation function (for non-linearity)
#         #self.relu= nn.leakyReLU() # <- add in ReLU activation function (for non-linearity)
#         # Can also put sigmoid in the model 
#         # This would mean you don't need to use it on the predictions
#         # self.sigmoid = nn.Sigmoid()

#     def forward(self, x):
#       # Intersperse the ReLU activation function between layers
#        return self.layer_3(self.relu(self.layer_2(self.relu(self.layer_1(x)))))

# model1 = BinaryClassifier().to(device)
# print(model1)

In [186]:
# # Build GW model with non-linear activation function
# from torch import nn

# featureNum = 5
# class BinaryClassifier(nn.Module):
#     def __init__(self):
#         super().__init__()
#         # This code works for SW HUC12
#         # self.layer_1 = nn.Linear(in_features=ncols, out_features=5) 
#         # self.layer_2 = nn.Linear(in_features=5, out_features=5)
#         # self.layer_3 = nn.Linear(in_features=5, out_features=1)

#         self.layer_1 = nn.Linear(in_features=ncols, out_features=featureNum) 
#         self.layer_2 = nn.Linear(in_features=featureNum, out_features=featureNum)
#         self.layer_3 = nn.Linear(in_features=featureNum, out_features=featureNum)
#         self.layer_4 = nn.Linear(in_features=featureNum, out_features=featureNum)
#         self.layer_5 = nn.Linear(in_features=featureNum, out_features=1)

#         self.relu = nn.ReLU(0.1) # <- add in ReLU activation function (for non-linearity)
#         #self.relu= nn.leakyReLU() # <- add in ReLU activation function (for non-linearity)
#         # Can also put sigmoid in the model 
#         # This would mean you don't need to use it on the predictions
#         # self.sigmoid = nn.Sigmoid()

#     def forward(self, x):
#       # Intersperse the ReLU activation function between layers
#        #return self.layer_3(self.relu(self.layer_2(self.relu(self.layer_1(x)))))
#        #return self.layer_4(self.layer_3(self.relu(self.layer_2(self.relu(self.layer_1(x))))))
#        return self.layer_5(self.layer_4(self.layer_3(self.relu(self.layer_2(self.relu(self.layer_1(x)))))))

# model1 = BinaryClassifier().to(device)
#print(model2)

In [187]:
import torch
import torch.nn as nn

class ImprovedBinaryClassifier(nn.Module):
    def __init__(self, input_dim=ncols, hidden_dim=64):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LeakyReLU(0.1),             # Prevents dead neurons
            nn.BatchNorm1d(hidden_dim),     # Stabilizes training
            
            # nn.Linear(hidden_dim, hidden_dim),
            # nn.LeakyReLU(0.1),             # Prevents dead neurons
            # nn.BatchNorm1d(hidden_dim),     # Stabilizes training

            nn.Linear(hidden_dim, hidden_dim),
            nn.LeakyReLU(0.1),
            nn.Dropout(0.2),                # Prevents overfitting
            
            nn.Linear(hidden_dim, 1)        # Outputs raw logits (No Sigmoid here - because signoid is used in next step when traning)
        )
        
    def forward(self, x):
        return self.network(x)

model1 = ImprovedBinaryClassifier().to(device)


In [188]:
# Setup loss and optimizer 
loss_fn = nn.BCEWithLogitsLoss()
#optimizer = torch.optim.SGD(model1.parameters(), lr=0.01)
optimizer1 = torch.optim.Adam(model1.parameters(), lr=0.01)
#optimizer2 = torch.optim.Adam(model2.parameters(), lr=0.01)

# Train the Model

In [189]:
# Fit the model (without cross-validation) on the entire dataset
# ~1 sec for SW
# 8 sec for GW

# Set the random seed for PyTorch (CPU)
seedvalue = 15 # 18 for SW
torch.manual_seed(seedvalue)

# Set the random seed for PyTorch (GPU / CUDA) if you use a graphics card
if torch.cuda.is_available():
  torch.cuda.manual_seed_all(seedvalue)

# Set the random seed for NumPy
np.random.seed(seedvalue)

# Set the random seed for Python's built-in random library
random.seed(seedvalue)

epochs = 1000

# Put all data on target device
X_train, y_train = X_train.to(device), y_train.to(device)
X_test, y_test = X_test.to(device), y_test.to(device)

for epoch in range(epochs):
    # 1. Forward pass
    y_logits = model1(X_train).squeeze()
    y_pred = torch.round(torch.sigmoid(y_logits)) # logits -> prediction probabilities -> prediction labels
    
    # 2. Calculate loss and accuracy
    loss = loss_fn(y_logits, y_train) # BCEWithLogitsLoss calculates loss using logits
    acc = accuracy_fn(y_true=y_train, 
                      y_pred=y_pred)
    
    # 3. Optimizer zero grad
    optimizer1.zero_grad()

    # 4. Loss backward
    loss.backward()

    # 5. Optimizer step
    optimizer1.step()

    ### Testing
    model1.eval()
    with torch.inference_mode():
      # 1. Forward pass
      test_logits = model1(X_test).squeeze()
      test_pred = torch.round(torch.sigmoid(test_logits)) # logits -> prediction probabilities -> prediction labels
      # 2. Calculate loss and accuracy
      test_loss = loss_fn(test_logits, y_test)
      test_acc = accuracy_fn(y_true=y_test,
                             y_pred=test_pred)

    # Print out what's happening
    if epoch % 100 == 0:
        print(f"Epoch: {epoch} | Loss: {loss:.5f}, Accuracy: {acc:.2f}% | Test Loss: {test_loss:.5f}, Test Accuracy: {test_acc:.2f}%")

Epoch: 0 | Loss: 0.68640, Accuracy: 53.87% | Test Loss: 1.12920, Test Accuracy: 59.27%


Epoch: 100 | Loss: 0.49962, Accuracy: 75.82% | Test Loss: 0.50736, Test Accuracy: 75.39%
Epoch: 200 | Loss: 0.47650, Accuracy: 77.30% | Test Loss: 0.47947, Test Accuracy: 76.94%
Epoch: 300 | Loss: 0.44385, Accuracy: 79.03% | Test Loss: 0.46320, Test Accuracy: 77.91%
Epoch: 400 | Loss: 0.42945, Accuracy: 79.61% | Test Loss: 0.45566, Test Accuracy: 78.48%
Epoch: 500 | Loss: 0.41321, Accuracy: 80.85% | Test Loss: 0.43778, Test Accuracy: 79.71%
Epoch: 600 | Loss: 0.40123, Accuracy: 82.01% | Test Loss: 0.42383, Test Accuracy: 80.54%
Epoch: 700 | Loss: 0.39726, Accuracy: 81.34% | Test Loss: 0.44008, Test Accuracy: 80.30%
Epoch: 800 | Loss: 0.37628, Accuracy: 83.57% | Test Loss: 0.41741, Test Accuracy: 81.66%
Epoch: 900 | Loss: 0.36331, Accuracy: 84.01% | Test Loss: 0.39346, Test Accuracy: 82.09%


# Model Evaluation Metrics
*PCC, Sensativity, Specificity, AUC

In [190]:
# Calculating Variable Importance (20 sec for GW)
from captum.attr import IntegratedGradients

# 1. Initialize the Integrated Gradients baseline tool
ig = IntegratedGradients(model1)

# 2. Compute attributions (turn off requires_grad on X if it's still on from before)
X.requires_grad_(False)
attributions, delta = ig.attribute(X, target=0, return_convergence_delta=True)

# 3. Aggregate importance across the batch
feature_importance_captum = attributions.abs().mean(dim=0)
#print("Captum Feature Importances:", feature_importance_captum)

# 2. Convert the PyTorch feature importance tensor to a NumPy array
importance_values = feature_importance_captum.cpu().numpy()

# 3. Create a Pandas Series mapping column names to their importance scores
feature_names = balanced_df.drop(columns=['Viol_Class']).columns
mapped_importance = pd.Series(importance_values, index=feature_names)

# 4. Sort and print the results
print("Captum Feature Importances with Field Names:")
print(mapped_importance.sort_values(ascending=False))

Captum Feature Importances with Field Names:
precip9120cat            3.900632
N_Surp_kgsqkm_2017cat    3.416026
WtDepCat                 1.659149
ElevCat                  1.412620
RockNCat                 1.291808
PopDen2010Cat            0.854997
tmean9120cat             0.770262
PctCrop2019Cat           0.583504
SandCat                  0.502314
N_TW2012Cat              0.429707
HydrlCondCat             0.404793
Fe2O3Cat                 0.371655
permcat                  0.266324
Hillslope_PctCat         0.232605
AgKffactCat              0.013085
dtype: float32


In [131]:
import torchmetrics

# Define your classification task ('binary', 'multiclass', or 'multilabel')
task = "binary"

# Initialize metrics
#sensitivity_metric = torchmetrics.classification.Recall(task=task)
#specificity_metric = torchmetrics.classification.Specificity(task=task)

# Get model predictions on the test set
model1.eval()
with torch.inference_mode():
    preds = torch.round(torch.sigmoid(model1(X_test))).squeeze()

# get target / observed response values
target = y_test

# Simulated model predictions (logits or probabilities) and ground truth targets
# preds  = torch.tensor([0, 1, 0, 1, 1, 0])
# target = torch.tensor([0, 1, 1, 0, 1, 0])

# Compute metrics
#sensitivity = sensitivity_metric(preds, target)
#specificity = specificity_metric(preds, target)

# Calculate True Positives, True Negatives, False Positives, False Negatives
TP = torch.sum((preds == 1) & (target == 1)).float()
TN = torch.sum((preds == 0) & (target == 0)).float()
FP = torch.sum((preds == 1) & (target == 0)).float()
FN = torch.sum((preds == 0) & (target == 1)).float()

PCC = (TP + TN) / (TP + TN + FP + FN)
sensitivity = TP / (TP + FN )
specificity = TN / (TN + FP )

print(f"PCC: {PCC.item():.4f}")
print(f"Sensitivity (True Positives): {sensitivity.item():.4f}")
print(f"Specificity (True Negatives): {specificity.item():.4f}")

PCC: 0.8156
Sensitivity (True Positives): 0.8248
Specificity (True Negatives): 0.8065


In [102]:
len(preds), len(target)
preds[0:20], target[0:20]

(tensor([1., 1., 0., 1., 1., 1., 0., 0., 1., 1., 0., 1., 0., 1., 0., 0., 1., 0.,
         1., 0.]),
 tensor([0., 1., 0., 1., 1., 1., 0., 0., 1., 1., 0., 1., 0., 1., 0., 0., 1., 0.,
         1., 0.]))

In [29]:
# AUC Calculation
from sklearn.metrics import roc_auc_score
import numpy as np

#()()()()()()()()()()()()()()()()()()()()()()()()()()()()()()()()()()()()()()()()
model = model1  # model 1 or model2, depending on which model you want to evaluate
#()()()()()()()()()()()()()()()()()()()()()()()()()()()()()()()()()()()()()()()()

model1.eval()
with torch.inference_mode():
    preds = torch.round(torch.sigmoid(model(X_test))).squeeze()

# get target / observed response values
target = y_test

preds2 = preds.detach().cpu().tolist()
target2 = target.detach().cpu().tolist()

len(preds), len(target), type(preds), type(preds2), target2[0:5], preds2[0:5]

# 2. Calculate the AUC Score
auc_score = roc_auc_score(target2, preds2)
print(f"Test AUC: {auc_score:.4f}")

Test AUC: 0.9153


# Save the Model

In [30]:
model_dir = future_dir + "/Models"
model_dir

'C:/Users/MPennino/OneDrive - Environmental Protection Agency (EPA)/Projects/OASES/Data/Future_NO3//Models'

In [31]:
#filename = '/torch_model_future_NO3_sw.pth'
filename = '/torch_model_future_NO3_gw.pth'

# #torch.save(model1.state_dict(), model_dir + filename)
torch.save(model1, model_dir + filename) # saves complete model
